# Transformadores Personalizados

## Pasos previos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
housing = pd.read_csv("./data/housing.csv") 
train_set, test_set = train_test_split(housing, test_size=0.2,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),
    random_state=42
    )
X_train = train_set.drop("median_house_value", axis=1) # Eliminar la columna de la variable dependiente
y_train = train_set["median_house_value"].copy() # Guardar la variable dependiente (etiquetas)

## Creación de Transformadores Personalizados

Para transformaciones que no requieren entrenamiento, basta con definir una función que reciba un array de NumPy y devuelva uno transformado, y pasarla a `FunctionTransformer` para crear un transformador personalizado. Estos transformadores permitirán crear objetos que se comportan como los de la librería `sklearn` y pueden usarse en sus *Pipelines*. Por ejemplo, para transformaciones logarítmicas:

In [ ]:
from sklearn.preprocessing import FunctionTransformerlog_transformer = FunctionTransformer(np.log, inverse_func=np.exp)log_pop = log_transformer.transform(X_train[["population"]])

o para combinar *características*:

In [ ]:
def column_ratio(X): # Transformador personalizado para calcular el ratio entre dos columnas
    return X[:, [0]] / X[:, [1]]
ratio_transformer = FunctionTransformer(column_ratio)    
X_train["rooms_per_household"] = ratio_transformer.fit_transform(X_train[['total_rooms', 'households']].values)
X_train[['rooms_per_household', 'total_rooms', 'households']].head()

El mismo ejemplo anterior puede definirse de forma más compacta usando una lambda (una función anónima):

In [ ]:
ratio_transformer = FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]])
X_train["rooms_per_household"] = ratio_transformer.transform(X_train[['total_rooms', 'households']].values)
X_train[['rooms_per_household', 'total_rooms', 'households']].head()

Cuando nuestra transformación requiere entrenamiento, podemos crear un transformador que tenga un método `fit` en el que se aprendan los parámetros necesarios y un método `transform` que aplique la transformación. Un transformador personalizado debe heredar de `BaseEstimator` (del que hereda los métodos `get_params` y `set_params`, necesarios para ajustar los parámetros de la transformación) y de `TransformerMixin` (que proporciona el método `fit_transform`).

Por ejemplo, definiendo un transformador que se comporte como `StandardScaler`:

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # ¡sin *args ni **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):  # y es requerido aunque no lo usemos
        X = check_array(X)  # verifica que X es un array con valores flotantes finitos
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # todos los estimadores almacenan esto en fit()
        return self  # ¡siempre devolver self!

    def transform(self, X):
        check_is_fitted(self)  # busca atributos aprendidos (con guión bajo al final)
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [ ]:
# Ejemplo de uso de un transformador personalizado
scaler = StandardScalerClone()
scaler.fit(X_train[["total_rooms"]])
scaler.transform(X_train[["total_rooms"]])

## Próximos pasos

Este notebook presentó los Transformadores Personalizados mediante `FunctionTransformer` y enfoques basados en clases. El pipeline de preprocesamiento se completa en:

- [e2e060 - Agrupamiento Espacial](e2e060_spatial_clustering.ipynb): transformador `ClusterSimilarity` para características geoespaciales usando K-means y el kernel RBF

El pipeline de preprocesamiento completo se consolida en [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py) para su reutilización en los notebooks de Entrenamiento del Modelo.